In [ ]:
import tkinter as tk
from tkinter import ttk
import heapq
import random
import time
import math

CELL = 25
WHITE="#ffffff"
BLACK="#2c3e50"
RED="#e74c3c"
GREEN="#2ecc71"
YELLOW="#f1c40f"
BLUE="#3498db"
PURPLE="#9b59b6"
BG="#ecf0f1"

root = tk.Tk()
root.title("Dynamic Pathfinding Agent")
root.geometry("1200x700")
root.configure(bg=BG)

ttk.Label(root,text="Dynamic Pathfinding Agent", font=("Arial",16,"bold")).pack(pady=10)

# Control frame top
control = ttk.Frame(root)
control.pack(pady=5, fill="x")

ttk.Label(control,text="Rows").grid(row=0,column=0,padx=5)
rows_entry = ttk.Entry(control,width=6)
rows_entry.insert(0,"20")
rows_entry.grid(row=0,column=1)

ttk.Label(control,text="Cols").grid(row=0,column=2,padx=5)
cols_entry = ttk.Entry(control,width=6)
cols_entry.insert(0,"20")
cols_entry.grid(row=0,column=3)

ttk.Label(control,text="Density").grid(row=0,column=4,padx=5)
density_entry = ttk.Entry(control,width=6)
density_entry.insert(0,"0.2")
density_entry.grid(row=0,column=5)

algo = tk.StringVar(value="A*")
heuristic_type = tk.StringVar(value="Manhattan")
dynamic_mode = tk.BooleanVar()

ttk.OptionMenu(control,algo,"A*","A*","Greedy").grid(row=0,column=6,padx=5)
ttk.OptionMenu(control,heuristic_type,"Manhattan","Manhattan","Euclidean").grid(row=0,column=7,padx=5)
ttk.Checkbutton(control,text="Dynamic Mode", variable=dynamic_mode).grid(row=0,column=8,padx=5)

# Main frame with canvas and right buttons
main_frame = ttk.Frame(root)
main_frame.pack(fill="both", expand=True, padx=10, pady=10)

# Scrollable canvas
canvas_frame = ttk.Frame(main_frame)
canvas_frame.pack(side="left", fill="both", expand=True)

canvas = tk.Canvas(canvas_frame,bg="white")
scroll_y = ttk.Scrollbar(canvas_frame,orient="vertical",command=canvas.yview)
scroll_x = ttk.Scrollbar(canvas_frame,orient="horizontal",command=canvas.xview)
canvas.configure(yscrollcommand=scroll_y.set, xscrollcommand=scroll_x.set)
scroll_y.pack(side="right",fill="y")
scroll_x.pack(side="bottom",fill="x")
canvas.pack(side="left",fill="both",expand=True)

# Buttons right side
button_frame = ttk.Frame(main_frame)
button_frame.pack(side="right", fill="y", padx=10, pady=10)

ttk.Button(button_frame,text="Create Grid", width=15, command=lambda: create_grid()).pack(pady=5)
ttk.Button(button_frame,text="Random Map", width=15, command=lambda: random_map()).pack(pady=5)
ttk.Button(button_frame,text="Start", width=15, command=lambda: start_search()).pack(pady=5)
ttk.Button(button_frame,text="Reset", width=15, command=lambda: reset()).pack(pady=5)

# Metrics at bottom
metrics = ttk.Label(root,text="",font=("Arial",11))
metrics.pack(pady=5)

grid=[]
ROWS=30
COLS=30
start=(0,0)
goal=(29,29)

# Create grid
def create_grid():
    global ROWS,COLS,grid,start,goal
    ROWS=int(rows_entry.get())
    COLS=int(cols_entry.get())
    canvas.config(scrollregion=(0,0,COLS*CELL,ROWS*CELL))
    grid=[[0]*COLS for _ in range(ROWS)]
    start=(0,0)
    goal=(ROWS-1,COLS-1)
    draw()

# Draw grid
def draw():
    canvas.delete("all")
    for r in range(ROWS):
        for c in range(COLS):
            color=WHITE if grid[r][c]==0 else BLACK
            canvas.create_rectangle(c*CELL,r*CELL,(c+1)*CELL,(r+1)*CELL, fill=color, outline="#bdc3c7")
    color_cell(start,BLUE)
    color_cell(goal,PURPLE)

# Color single cell
def color_cell(pos,color):
    r,c=pos
    canvas.create_rectangle(c*CELL,r*CELL,(c+1)*CELL,(r+1)*CELL, fill=color, outline="#bdc3c7")

# Toggle wall
def toggle_wall(event):
    r=int(canvas.canvasy(event.y)//CELL)
    c=int(canvas.canvasx(event.x)//CELL)
    if 0<=r<ROWS and 0<=c<COLS and (r,c)!=start and (r,c)!=goal:
        grid[r][c]=1-grid[r][c]
        draw()

canvas.bind("<Button-1>",toggle_wall)

# Random map
def random_map():
    density=float(density_entry.get())
    for r in range(ROWS):
        for c in range(COLS):
            if (r,c)!=start and (r,c)!=goal:
                grid[r][c]=1 if random.random()<density else 0
    draw()

# Heuristic
def heuristic(a,b):
    if heuristic_type.get()=="Manhattan":
        return abs(a[0]-b[0])+abs(a[1]-b[1])
    return math.sqrt((a[0]-b[0])**2+(a[1]-b[1])**2)

# Neighbors
def neighbors(node):
    r,c=node
    moves=[(1,0),(-1,0),(0,1),(0,-1)]
    return [(r+dr,c+dc) for dr,dc in moves if 0<=r+dr<ROWS and 0<=c+dc<COLS and grid[r+dr][c+dc]==0]

# Search
def search(start_pos):
    pq=[]
    heapq.heappush(pq,(0,start_pos))
    came={start_pos:None}
    g={start_pos:0}
    visited=set()
    while pq:
        _,cur=heapq.heappop(pq)
        if cur==goal: break
        visited.add(cur)
        color_cell(cur,RED)
        root.update()
        for nxt in neighbors(cur):
            new_cost=g[cur]+1
            if nxt not in g or new_cost<g[nxt]:
                g[nxt]=new_cost
                priority=new_cost+heuristic(nxt,goal) if algo.get()=="A*" else heuristic(nxt,goal)
                heapq.heappush(pq,(priority,nxt))
                came[nxt]=cur
                color_cell(nxt,YELLOW)
    path=[]
    node=goal
    while node in came and node is not None:
        path.append(node)
        node=came[node]
    return path[::-1],visited

# Spawn obstacle
def spawn_obstacle():
    if random.random()<0.05:
        r=random.randint(0,ROWS-1)
        c=random.randint(0,COLS-1)
        if (r,c)!=start and (r,c)!=goal:
            grid[r][c]=1
            color_cell((r,c),BLACK)

# Start search
def start_search():
    draw()
    start_time=time.time()
    current=start
    total_visited=0
    final_path=[]
    while current!=goal:
        path,visited=search(current)
        total_visited+=len(visited)
        if not path: break
        final_path=path
        for step in path[1:]:
            if dynamic_mode.get():
                spawn_obstacle()
                if grid[step[0]][step[1]]==1: break
            color_cell(step,GREEN)
            root.update()
            time.sleep(0.05)
            current=step
        else: continue
        continue
    end_time=time.time()
    cost=len(final_path) if final_path else 0
    metrics.config(text=f"Nodes Visited: {total_visited}   |   Path Cost: {cost}   |   Time: {round((end_time-start_time)*1000,2)} ms")

# Reset
def reset():
    create_grid()
    metrics.config(text="")

create_grid()
root.mainloop()